# Qwen-TTS 보이스 클로닝 실습 매뉴얼

부제: 구글 코랩에서 Qwen3-TTS Base 모델로 목소리를 복제해 보는 실습 교재

---

## 1. Qwen-TTS란?

Qwen-TTS는 Qwen 팀이 공개한 오픈소스 텍스트-투-스피치(TTS) 모델 시리즈다.
문장을 음성으로 바꾸는 기능뿐 아니라, 다음과 같은 기능을 지원한다.

- 다국어 음성 합성
- 자연어 지시를 통한 스타일 제어
- 스트리밍 기반 저지연 합성
- 참조 음성을 이용한 보이스 클로닝
- 미리 준비된 화자를 사용하는 TTS

공식 자료 기준으로 Qwen3-TTS는 다음 10개 언어를 지원한다.

Chinese, English, Japanese, Korean, German, French, Russian, Portuguese, Spanish, Italian

이 노트북은 그중 **보이스 클로닝**만 다룬다.

## 2. 이 노트북의 동작 방식

### 2.1 클로닝에 필요한 두 가지

Base 모델로 보이스 클로닝을 하려면 아래 두 입력이 필요하다.

- `ref_audio` — 복제할 목소리가 담긴 참조 음성 파일
- `ref_text` — 그 음성에 **실제로 들어 있는 문장**

둘이 어긋나면 품질이 눈에 띄게 떨어진다. 녹음한 그대로 받아써야 한다.

### 2.2 파일 이름 규칙

이 노트북은 **이름 하나만 입력하면** 오디오와 대본을 함께 찾아 읽는다.
그래서 두 파일의 이름(확장자 앞부분)이 서로 같아야 한다.

| 파일 | 역할 |
|---|---|
| `김남이_애국가1절.m4a` | 참조 음성 → `ref_audio` |
| `김남이_애국가1절.txt` | 그 음성의 대본 → `ref_text` |

`김남이_애국가1절` 이라고만 입력하면 위 두 파일을 자동으로 읽는다.
목소리를 추가하려면 같은 규칙으로 파일 쌍을 폴더에 넣기만 하면 된다.

### 2.3 Drive 폴더 구조

```
MyDrive/2026/clone_voice/
├── recored_voice/                 # 참조 음성 + 대본을 두는 곳
│   ├── 김남이_애국가1절.m4a
│   ├── 김남이_애국가1절.txt
│   └── 김남이_애국가1절_16k.wav   # 자동 생성 (변환 캐시)
└── output/                        # 생성된 음성이 저장되는 곳
```

### 2.4 전체 흐름

```
이름 입력 ──┬──▶ 이름.m4a ──ffmpeg──▶ 16kHz mono wav ──┐
            │                                          ├──▶ generate_voice_clone ──▶ 결과 wav
            └──▶ 이름.txt ─────────── ref_text ────────┤
                                                       │
생성할 문장 입력 ──────────── target_text ─────────────┘
```

참조 음성은 **한 번만** 불러오면 되고, 문장만 바꿔 가며 7번 셀을 반복 실행하면 된다.

## 3. Google Drive 연결

참조 음성과 대본을 Drive에서 읽어오므로 먼저 마운트한다.
실행하면 인증 창이 뜨고, 계정을 선택해 권한을 허용하면 된다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 환경 설치

- `qwen-tts` — 모델 로드와 합성을 담당하는 공식 패키지
- `soundfile` — wav 읽기/쓰기
- `ffmpeg` — m4a 같은 원본 녹음을 16kHz 모노 wav로 변환

설치 후 런타임 재시작을 요구하면 재시작하고, **3번 셀부터** 다시 실행한다.

In [ ]:
!pip install -U qwen-tts soundfile
!apt-get -y install ffmpeg

## 5. 모델 로드

Base 모델은 참조 음성을 따라 하는 용도의 체크포인트다.
GPU가 잡히면 `bfloat16`, 아니면 `float32`로 자동 설정된다.

첫 실행에서는 가중치(약 2.5GB)를 내려받으므로 몇 분 걸린다.
품질을 더 올리려면 `MODEL_ID`를 `Qwen/Qwen3-TTS-12Hz-1.7B-Base`로 바꾼다.
대신 VRAM을 더 쓰므로 무료 티어에서는 0.6B가 안전하다.

**런타임 → 런타임 유형 변경 → GPU** 설정을 먼저 확인할 것.

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print("DEVICE:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    dtype=DTYPE,
)

print("모델 로드 완료:", MODEL_ID)

## 6. 참조 음성 준비

### 6.1 설정과 헬퍼 함수

경로를 바꿔야 한다면 **이 셀의 `VOICE_DIR` / `OUTPUT_DIR`만** 수정하면 된다.

이 셀이 정의하는 것:

- `list_voices()` — 폴더에서 오디오와 `.txt` 짝이 맞는 이름만 찾아 목록으로 돌려준다
- `read_text()` — 한글 txt는 UTF-8과 CP949가 섞여 있어 순서대로 시도한다
- `to_wav()` — 16kHz 모노 wav로 변환한다. 이미 변환된 파일이 원본보다 최신이면 건너뛴다
- `load_reference()` — 이름 하나로 오디오와 대본을 함께 읽어 `(wav 경로, ref_text)`를 돌려준다

In [ ]:
import subprocess
from pathlib import Path
from datetime import datetime

import soundfile as sf
from IPython.display import Audio, display

# ── 설정 ────────────────────────────────────────────────
VOICE_DIR = Path("/content/drive/MyDrive/2026/clone_voice/recored_voice")
OUTPUT_DIR = Path("/content/drive/MyDrive/2026/clone_voice/output")
LANGUAGE = "Korean"
AUDIO_EXTS = [".m4a", ".mp3", ".wav", ".webm", ".ogg", ".flac"]
MIN_DURATION = 3.0
# ───────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def list_voices():
    """오디오와 .txt 짝이 모두 있는 이름만 돌려준다."""
    names = []
    for txt in sorted(VOICE_DIR.glob("*.txt")):
        if any((VOICE_DIR / f"{txt.stem}{ext}").exists() for ext in AUDIO_EXTS):
            names.append(txt.stem)
    return names


def read_text(path):
    """한글 txt 인코딩을 순서대로 시도한다."""
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return path.read_text(encoding=enc).strip()
        except UnicodeDecodeError:
            continue
    raise ValueError(f"'{path.name}' 인코딩을 인식하지 못했습니다. UTF-8로 저장해 주세요.")


def to_wav(src):
    """16kHz 모노 wav로 변환. 캐시가 원본보다 최신이면 건너뛴다."""
    dst = src.with_name(f"{src.stem}_16k.wav")
    if dst.exists() and dst.stat().st_mtime >= src.stat().st_mtime:
        return dst
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src), "-ac", "1", "-ar", "16000", str(dst)],
        check=True, capture_output=True,
    )
    return dst


def load_reference(name):
    """이름 하나로 참조 음성과 대본을 함께 읽는다."""
    name = name.strip()
    if not name:
        raise ValueError("참조 음성 이름을 입력해야 합니다.")

    # 확장자까지 붙여서 입력한 경우도 받아준다
    if Path(name).suffix.lower() in AUDIO_EXTS + [".txt"]:
        name = Path(name).stem

    audio = next(
        (VOICE_DIR / f"{name}{ext}" for ext in AUDIO_EXTS
         if (VOICE_DIR / f"{name}{ext}").exists()),
        None,
    )
    if audio is None:
        raise FileNotFoundError(
            f"'{name}' 오디오 파일을 찾을 수 없습니다.\n"
            f"  찾은 위치: {VOICE_DIR}\n"
            f"  사용 가능한 이름: {list_voices() or '없음'}"
        )

    txt = VOICE_DIR / f"{name}.txt"
    if not txt.exists():
        raise FileNotFoundError(
            f"'{name}.txt'가 없습니다. 참조 음성의 대본 파일이 함께 있어야 합니다."
        )

    ref_text = read_text(txt)
    if not ref_text:
        raise ValueError(f"'{txt.name}'이 비어 있습니다.")

    wav = to_wav(audio)
    data, sr = sf.read(wav)
    duration = len(data) / sr

    print(f"참조 음성  : {audio.name} ({duration:.2f}초)")
    print(f"참조 텍스트: {ref_text}")
    if duration < MIN_DURATION:
        print(f"경고: {MIN_DURATION}초 이상을 권장합니다. 짧으면 클로닝 품질이 떨어집니다.")

    return wav, ref_text


print("헬퍼 함수 준비 완료")
print("참조 음성 폴더:", VOICE_DIR)
print("출력 폴더    :", OUTPUT_DIR)

### 6.2 참조 음성 선택

폴더에서 사용 가능한 이름을 먼저 보여주고, 그중 하나를 입력받는다.
**확장자는 빼고** 이름만 입력한다. 예: `김남이_애국가1절`

목록이 비어 있으면 6.1의 `VOICE_DIR` 경로부터 확인한다.
매번 입력하기 번거로우면 `VOICE_NAME`에 직접 값을 넣어도 된다.

In [ ]:
print("사용 가능한 참조 음성")
for n in list_voices() or ["(없음 — VOICE_DIR 경로를 확인하세요)"]:
    print("  -", n)

VOICE_NAME = input("\n사용할 참조 음성 이름 (확장자 없이): ").strip()
# VOICE_NAME = "김남이_애국가1절"   # 고정해서 쓰려면 이 줄을 사용

REF_WAV, REF_TEXT = load_reference(VOICE_NAME)

print("\n참조 음성 미리듣기")
display(Audio(str(REF_WAV)))

## 7. 음성 생성

생성할 문장은 여기서 직접 입력받는다.
참조 음성은 6번에서 이미 불러왔으므로, **문장만 바꿔 가며 이 셀을 반복 실행**하면 된다.
모델을 다시 로드하거나 파일을 다시 변환하지 않는다.

결과는 `output` 폴더에 `이름_날짜시각.wav`로 저장되고, 아래에서 바로 재생된다.

In [ ]:
if "REF_WAV" not in globals():
    raise RuntimeError("먼저 6.2 셀에서 참조 음성을 불러오세요.")

target_text = input("생성할 문장을 입력하세요:\n> ").strip()
if not target_text:
    raise ValueError("생성할 문장을 입력해야 합니다.")

print("\n생성 중...")
wavs, out_sr = model.generate_voice_clone(
    text=target_text,
    language=LANGUAGE,
    ref_audio=str(REF_WAV),
    ref_text=REF_TEXT,
)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = OUTPUT_DIR / f"{VOICE_NAME}_{stamp}.wav"
sf.write(out_path, wavs[0], out_sr)

print("생성 완료:", out_path)
display(Audio(str(out_path)))

## 8. 자주 막히는 지점

| 증상 | 원인과 조치 |
|---|---|
| 사용 가능한 목록이 비어 있음 | `VOICE_DIR` 경로 오타, 또는 `.m4a`와 `.txt` 이름이 서로 다름 |
| `FileNotFoundError` | 확장자를 뺀 이름만 입력했는지 확인. 한글 파일명은 띄어쓰기·언더바까지 정확히 일치해야 함 |
| 대본이 깨져서 출력됨 | txt를 UTF-8로 다시 저장 |
| `ffmpeg` 오류 | 4번 셀의 `apt-get install ffmpeg`가 실행됐는지 확인 |
| 결과가 원본 목소리와 안 닮음 | `ref_text`가 녹음 내용과 정확히 일치하는지 확인. 3초 미만이거나 잡음이 많으면 품질이 떨어짐 |
| CUDA out of memory | 런타임 재시작 후 0.6B 모델 사용 |
| 생성이 매우 느림 | GPU 런타임이 아님. 런타임 유형을 GPU로 변경 |

### 참조 음성을 잘 만드는 요령

- 3초 이상, 10~20초 내외가 무난하다
- 배경 소음이 적고 한 사람만 말하는 구간을 쓴다
- 대본은 들리는 그대로 받아쓴다. 문장부호는 크게 중요하지 않다

### 주의

타인의 목소리를 복제할 때는 반드시 본인 동의를 받아야 한다.
동의 없는 음성 복제는 음성권 침해나 사기에 해당할 수 있다.